# GPU04 — خ۶ (F06): فرایند گاوسی (GP) روی L1 با GPyTorch

> بند 7.15 `doc/WBS-phase7-modeling.md` · اسپرینت C، ردیف «خ۶ کرنل: GP».

جدول 7.15.4 می‌گوید GP روی L1 (۷٬۵۷۹ نقطه) «در مرز» است و روی CPU ~۱۰ دقیقه
می‌گیرد. روی GPU همان کار ثانیه‌ای است — پس دلیل محدودکردن به L3 (۲۵۶ نقطه) از بین
می‌رود و GP روی **همان سطحی** اجرا می‌شود که همه‌ی خانواده‌های دیگر با آن سنجیده
شدند (بند 7.1.2). این خودش یکی از دلایل فرستادن این خانواده به GPU است.

**دو خروجی اجباری بند 7.15.6 که این نوت‌بوک تولید می‌کند:**

1. **جدول مقایسه‌ی ۷ ترکیب کرنل** (K1…K7 جدول 7.15.3) با درست‌نمایی حاشیه‌ای لگاریتمی.
2. **طول‌مقیاس ARD هر فیچر** — خروجی تفسیری: طول‌مقیاس کوچک ⇒ فیچر مهم. خودش یک
   روش انتخاب فیچر است (بند 7.5.4).

⭐ K7 (`gp_heteroscedastic`) واریانس نویز را تابع $\log Res$ می‌کند — **پاسخ مستقیم
به F06** (ناهم‌واریانسی تأییدشده، نسبت std چارک کوچک به بزرگ ۳.۰۳).

**بودجه‌ی هدف: ~۹۵ دقیقه.**

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [ ]:
!pip install -q gpytorch optuna mlflow tabulate

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [ ]:
MODE = "colab"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/input/phase7-bundle/gpu_bundle.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [ ]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.gpu_runner import load_l1
data = load_l1()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [ ]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [ ]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [ ]:
from pathlib import Path
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "colab"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

# reports/gpu/ را همین‌جا می‌سازیم — سلول‌های بعدی مستقیم CSV آن‌جا می‌نویسند،
# پیش از آنکه save_family_report/package_outputs بسازدش
Path("reports/gpu").mkdir(parents=True, exist_ok=True)


## سلول ۷-الف — R0: آزمایش دود

In [ ]:
from src.models.families import f06_kernel as fam
from src.models.gpu_runner import smoke_test

smoke = [smoke_test(fam.FITTERS[m], data, hyperparams={"n_iters": 40})
         for m in ["gp_quantile", "gp_heteroscedastic"]]

## سلول ۷-ب — ⭐ جدول اجباری ۷.۱۵.۶: مقایسه‌ی هفت ترکیب کرنل

هر ترکیب روی **هر ۵ fold رسمی** برازش می‌شود؛ هم pinball و هم درست‌نمایی حاشیه‌ای
لگاریتمی (LML) ثبت می‌شود. LML معیار انتخاب **درون‌مدلی** GP است و pinball معیار
پروژه — اگر این دو با هم نخوانند، خودش یک یافته است.

⏱ سنگین‌ترین سلول این نوت‌بوک (~۴۰ دقیقه). اگر session ناپایدار است `N_ITERS` را
کم کنید.

In [ ]:
import numpy as np, pandas as pd, time
from src.baselines import operational_metrics
from src.models.axes import TUNING_TAU
from src.models.gpu_runner import baseline_b3_per_fold

N_ITERS = 120
b3 = baseline_b3_per_fold(data.folds, TUNING_TAU)

rows = []
for combo in fam.KERNEL_COMBOS:
    het = combo == "K7_heteroscedastic"
    fitter = fam.FITTERS["gp_heteroscedastic" if het else "gp_quantile"]
    t0, pinballs, lmls, covs = time.time(), [], [], []
    try:
        for tr, te in data.folds:
            m = fitter.fit(tr, TUNING_TAU, kernel=("K5_full" if het else combo), n_iters=N_ITERS)
            met = operational_metrics(te, m.predict(te, TUNING_TAU), TUNING_TAU)
            pinballs.append(met["pinball"]); covs.append(met["coverage"]); lmls.append(m.lml)
        rows.append({"ترکیب کرنل": combo, "pinball": round(float(np.mean(pinballs)), 5),
                     "B3": round(b3["mean_pinball"], 5), "پوشش": round(float(np.mean(covs)), 4),
                     "LML (میانگین fold)": round(float(np.mean(lmls)), 2),
                     "ثانیه": round(time.time() - t0, 1)})
        print(f"  {combo:<24s} pinball={np.mean(pinballs):.5f}  LML={np.mean(lmls):8.2f}  "
              f"{time.time()-t0:5.1f}s")
    except Exception as e:
        rows.append({"ترکیب کرنل": combo, "pinball": float("nan"), "خطا": str(e)[:120]})
        print(f"  {combo:<24s} ❌ {type(e).__name__}: {str(e)[:100]}")

kernel_table = pd.DataFrame(rows).sort_values("pinball")
kernel_table.to_csv("reports/gpu/F06_kernel_comparison.csv", index=False)
kernel_table

## سلول ۷-ج — R2: تنظیم با بودجه‌ی زمانی

In [ ]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

studies = [
    run_gpu_study(fam.FITTERS["gp_quantile"], SPACES["gp_quantile"].fn, data,
                  family=fam.FAMILY, feature_set=fam.FEATURE_SET,
                  budget_minutes=22, compute=COMPUTE, seed=42),
    run_gpu_study(fam.FITTERS["gp_heteroscedastic"], SPACES["gp_heteroscedastic"].fn, data,
                  family=fam.FAMILY, feature_set=fam.FEATURE_SET,
                  budget_minutes=18, compute=COMPUTE, seed=42),
]

## سلول ۷-د — قهرمان + ACI + DM

GP قطعی است (بدون نمونه‌گیری تصادفی در برازش)، پس دو seed برای نشان‌دادن پایداری
بهینه‌سازی Adam کافی است — قاعده‌ی سه seed (A7) برای شبکه‌های عصبی نوشته شده.

In [ ]:
from src.models.gpu_runner import finalize_champion

best = min(studies, key=lambda s: s.best_pinball)
print(f"قهرمان: {best.model_id} (pinball={best.best_pinball:.5f})\n")
champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234),
                               compute=COMPUTE, run_aci=True)]

## سلول ۷-ه — ⭐ خروجی تفسیری اجباری: طول‌مقیاس ARD هر فیچر

طول‌مقیاس کوچک ⇒ خروجی با تغییر آن فیچر سریع عوض می‌شود ⇒ فیچر مهم است. این جدول
مستقیماً با اهمیت فیچر LightGBM (خ۲) قابل‌مقایسه است و یکی از خروجی‌های
گزارشی این خانواده است (بند 7.15.6).

In [ ]:
from pathlib import Path

stem = Path(f"models/gpu/F06/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
ard = reloaded.ard_lengthscales()
print(f"کرنل: {reloaded.config['kernel']} · LML={reloaded.lml:.2f} · "
      f"نویز ناهم‌واریانس: {reloaded.noise_model}")
ard.to_csv("reports/gpu/F06_ard_lengthscales.csv", index=False)
ard.head(20)

## سلول ۷-و — راستی‌آزمایی مدل ذخیره‌شده

⚠️ GP **پارامتری نیست**: بدون داده‌ی آموزش، هایپرپارامترها بی‌فایده‌اند — به همین
دلیل ماتریس آموزش هم داخل همان فایل ذخیره شده. این سلول ثابت می‌کند مدل بازخوانی‌شده
واقعاً پیش‌بینی می‌کند.

In [ ]:
import numpy as np
from src.models.axes import TAU_GRID

mean, sd = reloaded.posterior(data.folds[0][1])
print(f"پسین GP — میانگین={mean.mean():.5f} · انحراف معیار میانگین={sd.mean():.5f}")
print("τ | کوانتایل از همان پسین (یک برازش، همه‌ی τها):")
for t in TAU_GRID:
    print(f"{t:.2f} | {reloaded.predict(data.folds[0][1], t).mean():.5f}")
assert np.isfinite(mean).all() and np.isfinite(sd).all()
print("\n✅ مدل ذخیره‌شده قابل استفاده است")

## سلول ۷-ز — گزارش فارسی کامل

In [ ]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    "جدول مقایسه‌ی ۷ ترکیب کرنل (بند 7.15.3/7.15.6): `reports/gpu/F06_kernel_comparison.csv`.",
    "طول‌مقیاس ARD هر فیچر: `reports/gpu/F06_ard_lengthscales.csv` — خروجی تفسیری خانواده.",
    "GP روی L1 اجرا شد نه L3 — روی GPU محدودیت مقیاس‌پذیری بند 7.15.4 عملاً برطرف است.",
    "کوانتایل مستقیم از پسین گاوسی گرفته شد (مسیر Q4)، بدون آفست تجربی باقیمانده.",
    "K7 واریانس نویز را تابع log Res کرد — پاسخ مستقیم به F06؛ نتیجه‌اش در جدول کرنل.",
]
report = render_family_report("F06", "خ۶ — فرایند گاوسی روی L1 (GPyTorch، اجرای GPU)",
                              studies, champions, smoke, DEVICE, notes)
report += ("\n## مقایسه‌ی ترکیب‌های کرنل (بند 7.15.3)\n\n"
           + kernel_table.to_markdown(index=False)
           + "\n\n## ده فیچر با کوچک‌ترین طول‌مقیاس ARD\n\n"
           + ard.head(10).to_markdown(index=False) + "\n")
save_family_report("F06", report, "F06_gp_L1")
print(report)

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F06_gp_L1.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [ ]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F06_gp_L1", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```